# Async patterns comparison: asyncio vs trio vs anyio for I/O-bound DevOps tooling

## Purpose

DevOps tooling spends most of its wall-clock time waiting: probing HTTP endpoints, polling job status, copying artifacts, waiting on subprocesses. This notebook compares three widely used Python approaches to structuring that wait so many checks run concurrently instead of one after another. It covers the standard-library baseline (`asyncio`), the structured-concurrency style (`trio` nurseries), and the portable abstraction (`anyio` task groups), with runnable standard-library cells and gracefully degrading cells for the third-party packages.

## When to use each approach

- **asyncio** suits tooling that must run on the standard library with no extra install step. It is the default choice for small check scripts, CI helpers, and anything vendored into a minimal image.
- **trio-style nurseries** suit code where cancellation and error propagation must stay legible: a failed child cancels its siblings and the failure surfaces at the nursery exit point rather than leaking as a background exception.
- **anyio task groups** suit libraries and shared helpers that must run unmodified on more than one event-loop implementation. The same helper body works against either backend, so callers choose the runtime.

## Prerequisites

- A Python interpreter with the standard library only (every `asyncio` cell below runs as-is; cells that name third-party packages detect the import and skip with a message when the package is absent).
- No network access required: the simulated I/O below uses short sleeps standing in for endpoint probes.
- Familiarity with `async` / `await` function definitions and running a coroutine to completion.

In [ ]:
import asyncio
import time

# last_verified: 2026-09-20 · python n/a
# Pattern 1: asyncio fan-out with a semaphore cap.
# The semaphore bounds parallelism so a 200-host inventory scan
# does not open 200 simultaneous connections.

async def fake_probe(host: str, delay: float = 0.05) -> str:
    await asyncio.sleep(delay)
    return f"{host}: ok"

async def scan(hosts, limit: int = 5):
    sem = asyncio.Semaphore(limit)

    async def bounded(host):
        async with sem:
            return await fake_probe(host)

    return await asyncio.gather(*(bounded(h) for h in hosts))

hosts = [f"host-{i:02d}" for i in range(10)]
started = time.monotonic()
results = asyncio.run(scan(hosts))
elapsed = time.monotonic() - started
print(f"{len(results)} probes in {elapsed:.2f}s")
print(results[:3])

## Steps

1. Run the fan-out cell above and note the two moving parts: `asyncio.gather` for joining children and `asyncio.Semaphore` for bounding parallelism.
2. Run the timeout cell next and observe per-probe deadlines: a slow endpoint fails its own scope without stalling the whole scan.
3. Run the nursery-style and task-group cells and compare their structure against the `gather` version: one entry block, children started inside it, errors surfacing at the block exit.
4. Run the sequential-vs-concurrent comparison last and confirm the elapsed-time gap on the simulated workload.

In [ ]:
import asyncio

# Pattern 2: per-probe timeout so one slow endpoint cannot stall a scan.
# asyncio.wait_for wraps a single awaitable with a deadline; on expiry
# it cancels that awaitable and raises TimeoutError to the caller.

async def slow_probe() -> str:
    await asyncio.sleep(5.0)
    return "slow: ok"

async def main():
    try:
        return await asyncio.wait_for(slow_probe(), timeout=0.1)
    except TimeoutError:
        return "slow: timed out (expected)"

print(asyncio.run(main()))

In [ ]:
# Pattern 3: trio-style nursery (structured concurrency).
# Sketch only: runs when trio is installed, otherwise reports the skip.
# Children started inside the nursery block complete before the block
# exits; a child failure cancels its siblings and re-raises at the exit.

try:
    import trio  # type: ignore
except ImportError:
    trio = None

if trio is None:
    print("trio not installed -- showing the shape without executing")
    print("async with trio.open_nursery() as nursery:")
    print("    nursery.start_soon(probe, 'a')")
    print("    nursery.start_soon(probe, 'b')")
else:
    seen = []

    async def probe(name):
        await trio.sleep(0.05)
        seen.append(name)

    async def main():
        async with trio.open_nursery() as nursery:
            nursery.start_soon(probe, "a")
            nursery.start_soon(probe, "b")

    trio.run(main)
    print(sorted(seen))

In [ ]:
# Pattern 4: anyio task group (portable across backends).
# Same structure as a nursery, but the helper body runs unmodified on
# either backend. Skips gracefully when anyio is absent.

try:
    import anyio  # type: ignore
except ImportError:
    anyio = None

if anyio is None:
    print("anyio not installed -- showing the shape without executing")
    print("async with anyio.create_task_group() as tg:")
    print("    tg.start_soon(probe, 'a')")
    print("    tg.start_soon(probe, 'b')")
else:
    seen = []

    async def probe(name):
        await anyio.sleep(0.05)
        seen.append(name)

    async def main():
        async with anyio.create_task_group() as tg:
            tg.start_soon(probe, "a")
            tg.start_soon(probe, "b")

    anyio.run(main)
    print(sorted(seen))

In [ ]:
import asyncio
import time

# Pattern 5: sequential vs concurrent on the same simulated workload.
# Demonstrates the payoff for I/O-bound tooling: tasks overlap while
# waiting instead of stacking their latencies end to end.

async def fake_probe(host: str) -> str:
    await asyncio.sleep(0.05)
    return host

async def sequential(hosts):
    out = []
    for h in hosts:
        out.append(await fake_probe(h))
    return out

async def concurrent(hosts):
    return list(await asyncio.gather(*(fake_probe(h) for h in hosts)))

hosts = [f"host-{i:02d}" for i in range(10)]

t0 = time.monotonic()
asyncio.run(sequential(hosts))
t_seq = time.monotonic() - t0

t0 = time.monotonic()
asyncio.run(concurrent(hosts))
t_con = time.monotonic() - t0

print(f"sequential: {t_seq:.2f}s  concurrent: {t_con:.2f}s")
assert t_con < t_seq, "concurrent run should finish first"

## Verify

- Fan-out cell prints 10 probe results with an elapsed time well under the sequential total for the same workload.
- Timeout cell prints the timed-out sentinel rather than hanging on the multi-second sleep.
- Nursery / task-group cells either print the two completed names or the not-installed sketch; neither raises on a bare interpreter.
- Comparison cell prints both timings and passes its assertion that the concurrent run finishes first.

## Common errors

- Calling a coroutine without awaiting it (or without `asyncio.run`) returns a coroutine object and performs no work; the awaitable must be driven by a loop entry point.
- Unbounded fan-out against a real fleet exhausts file descriptors or trips rate limits; keep the semaphore cap and size it against the target service, not the host count.
- Mixing blocking calls (sleep, subprocess wait, file reads without a thread offload) inside a coroutine stalls every sibling sharing the loop; move blocking work to a worker thread or an async API.
- Timeouts placed around the whole `gather` instead of per probe turn one slow host into a full-scan failure; scope the deadline to the individual awaitable.

## References

- Standard-library asyncio task and timeout primitives (gather, Semaphore, wait_for).
- Nursery / task-group structured-concurrency pattern (single entry block, children joined at exit, failure cancels siblings).
- Follow-up: apply the semaphore plus per-probe timeout combination to an inventory scan helper and record the chosen parallelism cap.